# Anomaly Detection


In [7]:
import numpy as np
import random
import torch

from torch.utils.data import DataLoader
from torch.utils.data import (DataLoader, RandomSampler, SequentialSampler,
                              TensorDataset)
import torchvision.transforms as transforms

from torch import nn
import torch.nn.functional as F
from torch.autograd import Variable
import torchvision.models as models

from torch.optim import Adam, AdamW
import qqdm
from qqdm import format_str

train = np.load("data/trainingset.npy", allow_pickle=True) ## allow_pickle=True for loading list of arrays
test = np.load("data/testingset.npy", allow_pickle=True) 

print(train.shape)
print(test.shape)


def same_seeds(seed):
    # Python built-in random module
    random.seed(seed)
    # Numpy
    np.random.seed(seed)
    # Torch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

same_seeds(19530615)

class fcn_autoencoder(nn.Module):
    def __init__(self):
        super(fcn_autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(3*64*64, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 12),
            nn.ReLU(),
            nn.Linear(12, 3) ## latent space dimension
        )
        self.decoder = nn.Sequential(
            nn.Linear(3, 12),
            nn.ReLU(),
            nn.Linear(12, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 3*64*64),
            nn.Tanh() ## to get output in [0,1]
        )
    def forward(self, x):
        latent = self.encoder(x)
        output = self.decoder(latent)
        return output

class conv_autoencoder(nn.Module):
    def __init__(self):
        super(conv_autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3,12,4,2,1), ## out: 12x32x32
            nn.ReLU(),
            nn.Conv2d(12,24,4,2,1), ## out: 24x16x16
            nn.ReLU(),
            nn.Conv2d(24,48,4,2,1), ## out: 48x8x8
            nn.ReLU(),
            nn.Conv2d(48,96,4,2,1), ## out: 96x4x4
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(96,48,4,2,1), ## out: 48x8x8
            nn.ReLU(),
            nn.ConvTranspose2d(48,24,4,2,1), ## out: 24x16x16
            nn.ReLU(),
            nn.ConvTranspose2d(24,12,4,2,1), ## out: 12x32x32
            nn.ReLU(),
            nn.ConvTranspose2d(12,3,4,2,1), ## out: 3x64x64
            nn.Tanh() ## to get output in [0,1]
        )
    def forward(self, x):
        latent = self.encoder(x)
        output = self.decoder(latent)
        return output
    
class VAE(nn.Module):
    def __init__(self):
        super(VAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3,12,4,2,1), ## out: 12x32x32
            nn.ReLU(),
            nn.Conv2d(12,24,4,2,1), ## out: 24x16x16
            nn.ReLU()
        )
        self.enc_1 = nn.Sequential(
            nn.Conv2d(24,48,4,2,1), ## out: 48x8x8
            nn.ReLU()
        )
        self.enc_2 = nn.Sequential(
            nn.Conv2d(24,48,4,2,1), ## out: 48x8x8
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(48,24,4,2,1), ## out: 24x16x16
            nn.ReLU(),
            nn.ConvTranspose2d(24,12,4,2,1), ## out: 12x32x32
            nn.ReLU(),
            nn.ConvTranspose2d(12,3,4,2,1), ## out: 3x64x64
            nn.Tanh() ## to get output in [0,1]
        )
        
    def encode(self, x):
        h = self.encoder(x)
        z_mean = self.enc_1(h)
        z_log_var = self.enc_2(h)
        return z_mean, z_log_var
    
    def reparametrized(self, z_mean, z_log_var):
        std = z_log_var.mul(0.5).exp_() 
        if torch.cuda.is_available():
            eps = torch.cuda.FloatTensor(std.size()).normal_()
        else:
            eps = torch.FloatTensor(std.size()).normal_()
        eps = Variable(eps)
        return eps.mul(std).add_(z_mean)
    
    def decode(self, z):
        output = self.decoder(z)
        return output
    
    def forward(self, x):
        z_mean, z_log_var = self.encode(x)
        z = self.reparametrized(z_mean, z_log_var)
        output = self.decode(z)
        return output, z_mean, z_log_var
    
    def VAE_loss(self, recon_x, x, mu, logvar, criterion):
        loss = criterion(recon_x, x)
        KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) ## KL divergence between N(mu, sigma^2) and N(0,1)
        return loss + KLD
    
class ResNet(nn.Module):
    def __init__(self, fc_hidden_1=1024, fc_hidden_2=768, drop_p=0.3, CNN_embed_dim=256):
        super(ResNet, self).__init__()
        self.fc_hidden_1 = fc_hidden_1
        self.fc_hidden_2 = fc_hidden_2
        self.drop_p = drop_p
        self.CNN_embed_dim = CNN_embed_dim
        resnet = models.resnet18(pretrained=False)
        modules = list(resnet.children())[:-1] 
        self.resnet = nn.Sequential(*modules) ##(bs*512x1x1)
        self.fc1 = nn.Linear(resnet.fc.in_features, self.fc_hidden_1) ## 512 -> 1024
        self.bn1 = nn.BatchNorm1d(self.fc_hidden_1,momentum=0.01)
        self.fc2 = nn.Linear(self.fc_hidden_1, self.fc_hidden_2) ## 1024 -> 768
        self.bn2 = nn.BatchNorm1d(self.fc_hidden_2,momentum=0.01)
        self.fc3 = nn.Linear(self.fc_hidden_2, self.CNN_embed_dim) ## 768 -> 256
        
        self.fc4 = nn.Linear(self.CNN_embed_dim, self.fc_hidden_2) ## 256 -> 768
        self.bn4 = nn.BatchNorm1d(self.fc_hidden_2,momentum=0.01)
        self.fc5 = nn.Linear(self.fc_hidden_2, self.fc_hidden_1) ## 768 -> 1024
        self.bn5 = nn.BatchNorm1d(self.fc_hidden_1,momentum=0.01)
        self.relu = nn.ReLU(inplace=True)
        
        ##decode
        self.convtrans6 = nn.Sequential(
            nn.ConvTranspose2d(64,32,3,2,0),
            nn.BatchNorm2d(32,momentum=0.01),
            nn.ReLU(inplace=True)
        ) ## out: 32x4x4
        self.convtrans7 = nn.Sequential(
            nn.ConvTranspose2d(32,8,3,2,0),
            nn.BatchNorm2d(8,momentum=0.01),
            nn.ReLU(inplace=True)
        ) ## out: 8x8x8
        self.convtrans8 = nn.Sequential(
            nn.ConvTranspose2d(8,3,3,2,0),
            nn.BatchNorm2d(3,momentum=0.01),
            nn.Sigmoid()
        ) ## out: 3x16x16
        
    def encode(self, x):
        x = self.resnet(x) ##(512x1x1)
        x = x.view(x.size(0), -1) ##(512)
        if x.shape[0] > 1: 
            x = self.bn1(self.fc1(x)) ##(1024)
        else:
            x = self.fc1(x)
        if x.shape[0] > 1:
            x = self.bn2(self.fc2(x)) ##(768)
        else:
            x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x) ##(256)
        return x
    
    def decode(self, x):
        if x.shape[0] > 1:
            x = self.relu(self.bn4(self.fc4(x))) ##(768)
            x = self.relu(self.bn5(self.fc5(x))).view(-1, 64, 4, 4) ##(1024)
        else:
            x = self.relu(self.fc4(x))
            x = self.relu(self.fc5(x)).view(-1, 64, 4, 4) ##(1024)

        x = self.convtrans6(x) ##(bs*32*4*4)
        x = self.convtrans7(x) ##(bs*8*8*8)
        x = self.convtrans8(x) ##(bs*3*16*16)
        x = F.interpolate(x, size=(64,64), mode='bilinear', align_corners=True) ##(bs*3*64*64)
        return x

    def forward(self, x):
        latent = self.encode(x)
        output = self.decode(latent)
        return output
    
class CustomDataset(TensorDataset):
    def __init__(self, tensor):
        self.tensor = tensor
        if self.tensor.shape[-1] == 3:
            self.tensor = tensor.permute(0,3,1,2) ## (bs*3*64*64)
        self.transform = transforms.Compose([
            transforms.Lambda(lambda x: x.to(torch.float32)),
            transforms.Lambda(lambda x: 2.*x/255. - 1.), # normalize to [-1,1] range)
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])
    def __getitem__(self, index):
        return self.transform(self.tensor[index])
    def __len__(self):
        return len(self.tensor)
    
num_epochs = 50
batch_size = 10000
learning_rate = 1e-3

x = torch.from_numpy(train) ## (bs*64*64*3)
dataset = CustomDataset(x)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_type = "VAE"
model_class = {"fcn_autoencoder": fcn_autoencoder(), "conv_autoencoder": conv_autoencoder(), "VAE": VAE(), "ResNet": ResNet()}
model = model_class[model_type].to(device)
criterion = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=learning_rate)

best_loss = float('inf')
model.train()
qqdm_train = qqdm.qqdm(range(num_epochs), desc=format_str('bold', 'Description'))
for epoch in qqdm_train:
    tot_loss = list()
    for data in dataloader:
        if model_type in ["conv_autoencoder", "ResNet","VAE"]:
            img = data.float().to(device)
        else:
            img = data.float().to(device).view(data.size(0), -1)
        output = model(img)
        if model_type == "VAE":
            loss = model.VAE_loss(output[0], img, output[1], output[2], criterion)
        else:
            loss = criterion(output, img)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        tot_loss.append(loss.item())
    avg_loss = sum(tot_loss)/len(tot_loss)
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), f"{model_type}_best.pth")
    qqdm_train.set_infos({
        "Epoch": f"{epoch + 1}/{num_epochs}",
        "Loss": f"{avg_loss:.4f}",
    })


(100000, 64, 64, 3)
(19636, 64, 64, 3)


 Iters    Elapsed Time      Speed                                               
 0/50           -             -                                                 
Description   0.0% |                                                           | Iters    Elapsed Time      Speed    Epoch     Loss                             
 1/50   00:00:19<00:15:41  0.05it/s  1/50   27553.3468                          
Description   2.0% |█                                                          | Iters    Elapsed Time      Speed    Epoch   Loss                               
 2/50   00:00:37<00:15:11  0.05it/s  2/50   19.7809                             
Description   4.0% |██                                                         | Iters    Elapsed Time      Speed    Epoch   Loss                               
 3/50   00:00:57<00:14:58  0.05it/s  3/50   2.3724                              
Description   6.0% |███                                                        | Iters    Elapsed Time      Spee

In [10]:
import pandas as pd

eval_size = 200
data = torch.tensor(test, dtype=torch.float32) ## (200*64*64*3)
dataset = CustomDataset(data)
test_data = SequentialSampler(dataset)
test_dataloader = DataLoader(dataset, sampler=test_data, batch_size=eval_size)
eval_loss = nn.MSELoss(reduction='none')
checkpoint = torch.load(f"{model_type}_best.pth")
model.load_state_dict(checkpoint)
model.eval()
out_file = "predict.csv"

anonalaty = list()

with torch.no_grad():
    for i,data in enumerate(test_dataloader):
        if model_type in ["conv_autoencoder", "ResNet","VAE"]:
            img = data.float().to(device)
        else:
            img = data.float().to(device).view(data.size(0), -1)
        output = model(img)
        if model_type == "VAE":
            output = output[0]
        else:
            output = output
        if model_type == "fcn_autoencoder":
            loss = eval_loss(output, img).sum(-1)
        else:
            loss = eval_loss(output, img).sum([1,2,3])
            
        anonalaty.append(loss)
anonalaty = torch.cat(anonalaty, dim=0)
anonalaty = torch.sqrt(anonalaty).reshape(len(test),1).cpu().numpy()

df = pd.DataFrame(anonalaty, columns=["prediction"])
df.to_csv(out_file, index_label="id")
       
